# Emergency Vehicle Image Denoising using Convolutional Autoencoders

Image noise is a common problem in computer vision applications and can occur due to sensor limitations, poor lighting conditions, weather disturbances, transmission errors, or image compression. Excessive noise can degrade image quality and negatively impact downstream tasks such as object detection, classification, and recognition.

The objective of this project is to develop a Deep Learning-based image denoising system capable of reconstructing clean emergency vehicle images from noisy inputs. A Convolutional Autoencoder is used to learn meaningful image representations and remove noise while preserving important visual details.

The project follows a complete workflow including image loading, preprocessing, model development, training, and evaluation on unseen test images.


# Dataset Overview

The dataset consists of paired clean and noisy emergency vehicle images.

For each noisy image, a corresponding clean image is provided. During training, the noisy image is used as input while the clean image serves as the target output.

This paired-image setup allows the model to learn the mapping between noisy observations and their clean counterparts, making it suitable for supervised image denoising.


In [ ]:
import cv2
import os
import numpy as np

# Loading the Images

Before training the model, the clean and noisy images must be loaded into memory.

Since the dataset contains paired images, corresponding clean and noisy image files are loaded together to ensure that the model learns the correct reconstruction mapping.

Each image pair represents the same scene, with one image containing noise and the other representing the desired clean output.


In [ ]:
file_names = sorted(os.listdir("data\clean"))
print(file_names)

In [ ]:
clean_images = []
noisy_images = []

clean_dir = "data/clean"
noisy_dir = "data/noisy"

for file in file_names:
    # Load clean image
    clean_path = os.path.join(clean_dir, file)
    clean = cv2.imread(clean_path)

    # Load corresponding noisy image
    noisy_path = os.path.join(noisy_dir, file)
    noisy = cv2.imread(noisy_path)


    if clean is not None and noisy is not None:
        clean = cv2.cvtColor(clean, cv2.COLOR_BGR2RGB)
        clean_images.append(clean)

        noisy = cv2.cvtColor(noisy, cv2.COLOR_BGR2RGB)
        noisy_images.append(noisy)
    else:
        print(f"Warning: Could not load pair for {file}. Skipping.")

print(f"Successfully loaded {len(clean_images)} clean images and {len(noisy_images)} noisy images.")

# Image Preprocessing

Raw images cannot be directly fed into a neural network. Therefore, several preprocessing steps are performed:

* Converted list of images into array with datatype = float16
* Normalizing pixel intensities to the range [0, 1]

Normalization helps stabilize training and improves optimization efficiency by ensuring a consistent input scale across all images.


In [ ]:
clean_images = np.array(clean_images, dtype=np.float16)
noisy_images = np.array(noisy_images, dtype=np.float16)

In [ ]:
clean_images.shape

In [ ]:
noisy_images.shape

In [ ]:
noisy_images = noisy_images / noisy_images.max()
clean_images = clean_images / clean_images.max()

# Building the Convolutional Autoencoder

A Convolutional Autoencoder is used to perform image denoising.

The architecture consists of two primary components:

### Encoder

The encoder progressively compresses the input image into a compact latent representation by extracting meaningful visual features while discarding irrelevant information.

### Decoder

The decoder reconstructs a clean image from the latent representation, attempting to preserve important image structures while removing noise.

The network is trained end-to-end to minimize the reconstruction error between the predicted image and the corresponding clean target image.


In [ ]:
from keras.models import Model
from keras.layers import *
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

In [ ]:
inputs = Input(shape=(224, 224, 3))

x = Conv2D(32, (3, 3), padding='same')(inputs)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = MaxPooling2D((2, 2), padding='same')(x)

x = Conv2D(64, (3,3), padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = MaxPooling2D((2, 2), padding='same')(x)

x = Conv2D(128, (3,3), padding='same', activation='relu')(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
encoded = MaxPooling2D((2, 2), padding='same')(x)

x = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same', activation='relu')(encoded)
x = Conv2D(128, (3,3), padding='same', activation='relu')(x)

x = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same', activation='relu')(x)
x = Conv2D(64, (3,3), padding='same', activation='relu')(x)

x = Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same', activation='relu')(x)
x = Conv2D(32, (3,3), padding='same', activation='relu')(x)

outputs = Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)

autoencoder = Model(inputs, outputs)

# Model Summary

The model summary provides a layer-by-layer overview of the architecture, including the number of trainable parameters and the dimensional transformations occurring throughout the encoder and decoder.

Understanding the architecture helps evaluate model complexity and computational requirements.


# Model Compilation

The model is compiled using the Adam optimizer and a reconstruction loss function.

The reconstruction loss measures the difference between the predicted denoised image and the corresponding clean image. By minimizing this loss, the model gradually learns how to remove noise while preserving meaningful image content.


In [ ]:
autoencoder.compile(optimizer='adam', loss='mae')

In [ ]:
Early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    mode='min',
    restore_best_weights=True
)

Reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    mode='min'
)

model_checkpoint = ModelCheckpoint(
    'autoencoder_model.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min'
)

# Training the Autoencoder

During training, noisy images are provided as inputs while clean images are used as targets.

The objective of training is to learn a transformation that converts noisy observations into clean reconstructions.

As training progresses, the model learns to identify noise patterns and suppress them while preserving important image features such as edges, shapes, and textures.


In [ ]:
autoencoder.fit(noisy_images, clean_images, epochs=100, batch_size=32, validation_split=0.2, callbacks=[Early_stopping, Reduce_lr, model_checkpoint])

In [ ]:
from keras.models import load_model

autoencoder = load_model("models\denoising_autoencoder.keras")

In [ ]:
autoencoder.save_weights("models\denoising_autoencoder.weights.h5")

In [ ]:
image = cv2.imread("assets\test_sample_noisy_images\1043.jpg")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image = np.array(image, dtype=np.float16)
image = image / 255.0

In [ ]:
pred = autoencoder.predict(image.reshape(1, 224, 224, 3))

# Testing on Unseen Images

After training, the model is evaluated on previously unseen noisy images from the test dataset.

The purpose of testing is to assess the model's ability to generalize and successfully denoise images that were not encountered during training.

The resulting denoised images are compared visually against their noisy inputs to evaluate reconstruction quality.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(image.astype(np.float32))
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(pred[0])
plt.title('Denoised Image')
plt.axis('off')

plt.show()

# Results and Observations

The Convolutional Autoencoder successfully learned to reduce noise in emergency vehicle images.

The denoised outputs preserve the overall structure and appearance of the vehicles while removing a significant portion of the random noise present in the input images.

Although some fine details may become slightly blurred during reconstruction, the model demonstrates effective noise suppression and meaningful image recovery.


# Conclusion

In this project, a Convolutional Autoencoder was developed for image denoising using paired clean and noisy emergency vehicle images.

The model successfully learned latent image representations and reconstructed cleaner versions of noisy inputs. This project demonstrates the effectiveness of Autoencoders for image restoration tasks and provides a foundation for more advanced techniques such as Denoising Autoencoders, Variational Autoencoders (VAEs), and modern image restoration architectures.

Future improvements may include deeper architectures, skip connections, perceptual loss functions, and advanced image restoration models for improved reconstruction quality.
